# E2 · Finite-shot energy estimation

## 1. Purpose and design

E2 measures energy RMSE for frozen ICS, SOCP and VGMC designs. Exact and CISD designs are sampled on the reference state. Joint bitstrings supply the Pauli outcomes within each group. Integer allocations reserve one shot for each nonzero coefficient block and distribute the remaining budget by the recorded shot fractions.

Each budget uses 300 repetitions. RMSE intervals use 2,000 percentile bootstrap samples. The figures show RMSE in mHa and the 1.6 mHa chemical-accuracy reference. The full view uses logarithmic axes. The zoomed view covers 0.9–3.0 mHa.

The molecular Hamiltonians use the authors' fermionic files and the Bravyi–Kitaev transform. [Hamiltonian sources](../docs/sources.md#molecular-hamiltonians)

## 2. Imports and settings

`MODE = "analysis_only"` processes the supplied records. `MODE = "full"` executes the configured tasks in `RUN` and analyzes their commits. `MODE = "render"` loads an existing analysis. `RUN_MODE` selects fresh execution, compatible resume, verified reuse or automatic selection.


In [ ]:
from pathlib import Path
import logging

from IPython.display import Image, display
from vgmc.config import SamplingConfig
from vgmc.data import read_json
from vgmc.workflow import analyze, load_analysis, render, run_experiment

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "DATA_MANIFEST.json").is_file())
MODE = "analysis_only"
RUN_MODE = "fresh"
MOLECULES = ['LiH', 'BeH2', 'H2O', 'NH3']
RUN = ROOT / "outputs" / "E2" / "run-001"
OUTPUT = ROOT / "outputs" / "E2" / "report"
ANALYSIS = OUTPUT / "analysis"
CONFIG = SamplingConfig.from_mapping(
    read_json(ROOT / "configs" / "E2.json"))
logging.basicConfig(level=logging.INFO, format="%(message)s")
COVARIANCES = ["exact", "cisd"]
METHODS = ["ICS", "SOCP", "VGMC"]
WORKERS = 1


## 3. Run, collect and display

The analysis stores the selected rows and source hashes. Rendering reads that saved analysis.


In [ ]:
if MODE == "full":
    run_experiment("E2", ROOT, RUN, molecules=MOLECULES,
                   covariances=COVARIANCES, methods=METHODS, workers=WORKERS,
                   config=CONFIG, mode=RUN_MODE, allow_compute=True)
    analyze("E2", ROOT, ANALYSIS, run=RUN, molecules=MOLECULES)
elif MODE == "analysis_only":
    analyze("E2", ROOT, ANALYSIS, molecules=MOLECULES)
elif MODE != "render":
    raise ValueError("MODE must be analysis_only, full or render")

results = load_analysis("E2", ANALYSIS)
artifacts = render("E2", ANALYSIS, OUTPUT)


In [ ]:
display(results[["molecule", "covariance", "method", "budget", "rmse", "rmse_ci95"]].head(12))
for path in artifacts:
    if path.suffix == ".png":
        display(Image(filename=str(path)))
